# Project 1 – Incremental Spark ETL

Course: Big Data Management  
Student: BULAND KUMAR PRADHAN 

Goal:  
Build a Spark ETL pipeline that processes NYC Taxi trip records incrementally, applies cleaning rules, enriches the data with zone information, and writes a clean Parquet dataset.

In [1]:
import os

for root, dirs, files in os.walk(".", topdown=True):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

./
    Project1.ipynb
    .ipynb_checkpoints/
        Project1-checkpoint.ipynb
    data/
        taxi_zone_lookup.parquet
        inbox/
            yellow_tripdata_2025-01.parquet
            yellow_tripdata_2025-02.parquet
        outbox/
    state/
        manifest.json


## 1. Spark Session Setup

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

In [3]:
spark.version

'4.1.0'

## 2. Manifest State Management

The manifest tracks which input files have already been processed.
This ensures the pipeline processes only new files on each run.

In [4]:
import json
import os

MANIFEST_PATH = "state/manifest.json"

def load_manifest(path):
    if not os.path.exists(path):
        return {"processed_files": {}}
    with open(path, "r") as f:
        return json.load(f)

manifest = load_manifest(MANIFEST_PATH)

print("Manifest content:")
print(manifest)

Manifest content:
{'processed_files': {}}


## 3. Detect New Files

The pipeline reads files from `data/inbox` and compares them with
the manifest to determine which files need to be processed.

In [5]:
INBOX_DIR = "data/inbox"

def list_parquet_files(folder):
    files = []
    for name in os.listdir(folder):
        if name.endswith(".parquet"):
            files.append(name)
    return sorted(files)

inbox_files = list_parquet_files(INBOX_DIR)

print("Inbox files:")
for f in inbox_files:
    print("-", f)

Inbox files:
- yellow_tripdata_2025-01.parquet
- yellow_tripdata_2025-02.parquet


In [6]:
processed_files = manifest.get("processed_files", {})
new_files = [f for f in inbox_files if f not in processed_files]

print("New files detected:")
for f in new_files:
    print("-", f)

New files detected:
- yellow_tripdata_2025-01.parquet
- yellow_tripdata_2025-02.parquet


In [7]:
from pyspark.sql.functions import input_file_name, lit, regexp_extract
from datetime import datetime, timezone

# Full paths of new files
new_paths = [f"{INBOX_DIR}/{f}" for f in new_files]

print("Reading files:")
for p in new_paths:
    print("-", p)

df_raw = spark.read.parquet(*new_paths)

# Add metadata columns
ingested_time = datetime.now(timezone.utc).isoformat()

df_raw = (
    df_raw
    .withColumn("source_file_path", input_file_name())
    .withColumn(
        "source_file",
        regexp_extract(input_file_name(), r"([^/]+$)", 1)  # extract filename only
    )
    .withColumn("ingested_at", lit(ingested_time))
)

print("Row count of new data:", df_raw.count())

df_raw.select("source_file").distinct().show(truncate=False)

Reading files:
- data/inbox/yellow_tripdata_2025-01.parquet
- data/inbox/yellow_tripdata_2025-02.parquet
Row count of new data: 7052769
+-------------------------------+
|source_file                    |
+-------------------------------+
|yellow_tripdata_2025-01.parquet|
|yellow_tripdata_2025-02.parquet|
+-------------------------------+



## 4. Read New Trip Data

In [8]:
from pyspark.sql.functions import col, date_format, to_date, unix_timestamp

df_typed = (
    df_raw
    .withColumn("pickup_ts", col("tpep_pickup_datetime"))
    .withColumn("dropoff_ts", col("tpep_dropoff_datetime"))
    .withColumn("pickup_date", to_date(col("pickup_ts")))
    .withColumn("pickup_month", date_format(col("pickup_ts"), "yyyy-MM"))
    .withColumn(
        "trip_duration_minutes",
        (unix_timestamp("dropoff_ts") - unix_timestamp("pickup_ts")) / 60
    )
)

print("New row count after typing:", df_typed.count())

df_typed.select(
    "pickup_ts",
    "dropoff_ts",
    "pickup_date",
    "pickup_month",
    "trip_duration_minutes"
).show(5, truncate=False)

New row count after typing: 7052769
+-------------------+-------------------+-----------+------------+---------------------+
|pickup_ts          |dropoff_ts         |pickup_date|pickup_month|trip_duration_minutes|
+-------------------+-------------------+-----------+------------+---------------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|2025-01-01 |2025-01     |8.35                 |
|2025-01-01 00:32:40|2025-01-01 00:35:13|2025-01-01 |2025-01     |2.55                 |
|2025-01-01 00:44:04|2025-01-01 00:46:01|2025-01-01 |2025-01     |1.95                 |
|2025-01-01 00:14:27|2025-01-01 00:20:01|2025-01-01 |2025-01     |5.566666666666666    |
|2025-01-01 00:21:34|2025-01-01 00:25:06|2025-01-01 |2025-01     |3.533333333333333    |
+-------------------+-------------------+-----------+------------+---------------------+
only showing top 5 rows


In [9]:
df_typed.filter(col("trip_duration_minutes") < 0).count()

217

## 5. Scenario: Passenger Count Imputation

If `passenger_count` is null or zero, it is replaced with the median
passenger count for the same calendar month within the same source file.

In [10]:
df_typed.filter(
    (col("passenger_count").isNull()) | (col("passenger_count") == 0)
).count()

1393493

In [11]:
from pyspark.sql.functions import expr

# Only valid passenger_count rows
valid_df = df_typed.filter(col("passenger_count") > 0)

# Compute median per file and month
median_df = (
    valid_df
    .groupBy("source_file", "pickup_month")
    .agg(
        expr("percentile_approx(passenger_count, 0.5, 10000)").alias("median_passenger_count")
    )
)

median_df.show()

+--------------------+------------+----------------------+
|         source_file|pickup_month|median_passenger_count|
+--------------------+------------+----------------------+
|yellow_tripdata_2...|     2024-12|                     1|
|yellow_tripdata_2...|     2025-01|                     1|
|yellow_tripdata_2...|     2025-02|                     1|
|yellow_tripdata_2...|     2025-02|                     1|
|yellow_tripdata_2...|     2025-01|                     1|
|yellow_tripdata_2...|     2025-03|                     1|
+--------------------+------------+----------------------+



In [12]:
# Join medians back to main dataframe
df_joined = df_typed.join(
    median_df,
    on=["source_file", "pickup_month"],
    how="left"
)

# Apply imputation rule
from pyspark.sql.functions import when

df_imputed = (
    df_joined
    .withColumn(
        "passenger_count_clean",
        when(
            (col("passenger_count").isNull()) | (col("passenger_count") == 0),
            col("median_passenger_count")
        ).otherwise(col("passenger_count"))
    )
)

In [13]:
df_imputed.filter(
    (col("passenger_count_clean").isNull()) | 
    (col("passenger_count_clean") == 0)
).count()

0

In [14]:
from pyspark.sql.functions import sum as spark_sum

df_stats = (
    df_joined
    .withColumn(
        "was_imputed",
        ((col("passenger_count").isNull()) | (col("passenger_count") == 0)).cast("int")
    )
    .groupBy("source_file", "pickup_month", "median_passenger_count")
    .agg(
        spark_sum("was_imputed").alias("imputed_rows"),
        )
    .orderBy("source_file", "pickup_month")
)

df_stats.show(truncate=False)

+-------------------------------+------------+----------------------+------------+
|source_file                    |pickup_month|median_passenger_count|imputed_rows|
+-------------------------------+------------+----------------------+------------+
|yellow_tripdata_2025-01.parquet|2024-12     |1                     |0           |
|yellow_tripdata_2025-01.parquet|2025-01     |1                     |564805      |
|yellow_tripdata_2025-01.parquet|2025-02     |1                     |0           |
|yellow_tripdata_2025-02.parquet|2025-01     |1                     |0           |
|yellow_tripdata_2025-02.parquet|2025-02     |1                     |828688      |
|yellow_tripdata_2025-02.parquet|2025-03     |1                     |0           |
+-------------------------------+------------+----------------------+------------+



## 6. Data Cleaning

Invalid records such as negative trip durations are removed.

In [15]:
df_clean = df_imputed.filter(col("trip_duration_minutes") >= 0)

print("Rows before cleaning:", df_imputed.count())
print("Rows after removing negative durations:", df_clean.count())

Rows before cleaning: 7052769
Rows after removing negative durations: 7052552


## 7. Deduplication

Duplicate trips are removed using a key consisting of:

- VendorID
- pickup_ts
- dropoff_ts
- PULocationID
- DOLocationID
- trip_distance

In [16]:
dedup_key = [
    "VendorID",
    "pickup_ts",
    "dropoff_ts",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

rows_before = df_clean.count()

df_dedup = df_clean.dropDuplicates(dedup_key)

rows_after = df_dedup.count()

print("Rows before dedup:", rows_before)
print("Rows after dedup:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

Rows before dedup: 7052552
Rows after dedup: 6951037
Duplicates removed: 101515


In [17]:
lookup = spark.read.parquet("data/taxi_zone_lookup.parquet")

# pickup zone join
df_enriched = df_dedup.join(
    lookup.withColumnRenamed("LocationID", "PULocationID")
          .withColumnRenamed("Zone", "pickup_zone"),
    on="PULocationID",
    how="left"
)

# dropoff zone join
df_enriched = df_enriched.join(
    lookup.withColumnRenamed("LocationID", "DOLocationID")
          .withColumnRenamed("Zone", "dropoff_zone"),
    on="DOLocationID",
    how="left"
)

print("Rows after enrichment:", df_enriched.count())
df_enriched.select("pickup_zone", "dropoff_zone").show(5, truncate=False)

Rows after enrichment: 6951037
+-----------------+-----------------------------+
|pickup_zone      |dropoff_zone                 |
+-----------------+-----------------------------+
|LaGuardia Airport|Long Island City/Queens Plaza|
|Central Harlem   |Hollis                       |
|Lower East Side  |Williamsburg (South Side)    |
|East Chelsea     |Yorkville West               |
|Yorkville West   |Lenox Hill West              |
+-----------------+-----------------------------+
only showing top 5 rows


## 8. Data Enrichment

Pickup and dropoff zones are added using the taxi zone lookup table.

In [18]:
df_final = df_enriched.select(
    "pickup_ts",
    "dropoff_ts",
    "PULocationID",
    "DOLocationID",
    "pickup_zone",
    "dropoff_zone",
    col("passenger_count_clean").alias("passenger_count"),
    "trip_distance",
    "trip_duration_minutes",
    "pickup_date",
    "source_file",
    "ingested_at"
)

print("Final dataset rows:", df_final.count())
df_final.printSchema()

Final dataset rows: 6951037
root
 |-- pickup_ts: timestamp_ntz (nullable = true)
 |-- dropoff_ts: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: string (nullable = false)



## 9. Write Final Dataset

In [19]:
OUTPUT_PATH = "data/outbox/trips_enriched.parquet"

df_final.write.mode("overwrite").parquet(OUTPUT_PATH)

print("Output written to:", OUTPUT_PATH)

Output written to: data/outbox/trips_enriched.parquet


In [20]:
check_df = spark.read.parquet(OUTPUT_PATH)

print("Rows in written dataset:", check_df.count())
check_df.show(5)

Rows in written dataset: 6951037
+-------------------+-------------------+------------+------------+--------------+--------------+---------------+-------------+---------------------+-----------+--------------------+--------------------+
|          pickup_ts|         dropoff_ts|PULocationID|DOLocationID|   pickup_zone|  dropoff_zone|passenger_count|trip_distance|trip_duration_minutes|pickup_date|         source_file|         ingested_at|
+-------------------+-------------------+------------+------------+--------------+--------------+---------------+-------------+---------------------+-----------+--------------------+--------------------+
|2025-01-10 15:32:46|2025-01-10 15:34:20|           1|           1|Newark Airport|Newark Airport|              1|          0.0|   1.5666666666666667| 2025-01-10|yellow_tripdata_2...|2026-03-08T07:31:...|
|2025-01-14 16:43:08|2025-01-14 16:44:37|           1|           1|Newark Airport|Newark Airport|              1|          0.0|   1.4833333333333334| 2

## 10. Update Manifest

In [21]:
import json

for f in new_files:
    manifest["processed_files"][f] = {
        "processed_at": ingested_time
    }

with open(MANIFEST_PATH, "w") as fp:
    json.dump(manifest, fp, indent=2)

print("Manifest updated.")
print(json.dumps(manifest, indent=2))

Manifest updated.
{
  "processed_files": {
    "yellow_tripdata_2025-01.parquet": {
      "processed_at": "2026-03-08T07:31:08.831650+00:00"
    },
    "yellow_tripdata_2025-02.parquet": {
      "processed_at": "2026-03-08T07:31:08.831650+00:00"
    }
  }
}


In [22]:
processed_files = manifest.get("processed_files", {})
new_files = [f for f in inbox_files if f not in processed_files]

print("New files detected:")
for f in new_files:
    print("-", f)

New files detected:
